In [1]:
import json
import logging
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from Data.features.VKFeatureExtractor import VKFeaturesExtractor
import os

In [2]:
load_dotenv("/Users/olgagavrilenko/DataspellProjects/Course-work/Scripts/token.env")

token = os.getenv("ACCESS_TOKEN_VK")
logging.basicConfig(level=logging.INFO)

INPUT_JSON = Path("Passings 18.12.2024 (4).json")       
OUTPUT_DIR = Path("prepared_from_vk")
TARGET_MAP = {
    "result/0/Экстраверсия–интроверсия": "Экстраверсия–интроверсия",
    "result/0/Привязанность–обособленность": "Привязанность–обособленность",
    "result/0/Самоконтроль–импульсивность": "Самоконтроль–импульсивность",
    "result/0/Эмоциональная_устойчивость–эмоциональная_неустойчивость": "Эмоциональная_устойчивость–эмоциональная_неустойчивость",
    "result/0/Экспрессивность–практичность": "Экспрессивность–практичность",
}
MAX_WORKERS = 3

In [3]:
def load_raw_json_records(path: str | Path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path.resolve()}")

    text = path.read_text(encoding="utf-8", errors="replace").strip()
    data = json.loads(text)

    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        return [data]

    raise ValueError("Неожиданный формат JSON")


def extract_targets_from_record(record: dict, target_map: dict[str, str]) -> dict | None:
    vk_id = record.get("vk_id")
    completion_date = record.get("completion_date")
    result = record.get("result")

    if vk_id is None or result is None:
        return None

    if isinstance(result, str):
        try:
            result = json.loads(result)
        except Exception:
            return None

    if not isinstance(result, list) or len(result) == 0:
        return None

    first_block = result[0]
    if not isinstance(first_block, dict):
        return None

    row = {
        "vk_id": vk_id,
        "completion_date": completion_date,
    }

    for out_col, source_key in target_map.items():
        row[out_col] = first_block.get(source_key)

    return row


def load_targets_from_json(path: str | Path, target_map: dict[str, str]) -> pd.DataFrame:
    records = load_raw_json_records(path)

    rows = []
    for rec in records:
        row = extract_targets_from_record(rec, target_map)
        if row is not None:
            rows.append(row)

    if not rows:
        raise ValueError("Не удалось извлечь ни одной записи с таргетами.")

    df_targets = pd.DataFrame(rows)

    df_targets["completion_date"] = pd.to_datetime(
        df_targets["completion_date"],
        errors="coerce"
    )

    target_cols = list(target_map.keys())

    df_targets = df_targets.dropna(subset=["vk_id"]).copy()
    df_targets = df_targets.dropna(subset=target_cols, how="all").copy()

    # как в репозитории: последняя запись по completion_date для каждого vk_id
    df_targets = df_targets.sort_values("completion_date")
    df_targets = df_targets.drop_duplicates(subset=["vk_id"], keep="last").copy()

    df_targets = df_targets.drop(columns=["completion_date"])
    return df_targets

In [7]:
import logging
from datetime import datetime
import vk_api

NODE_FEATURES_FILE = OUTPUT_DIR / "node_features_ready.csv"
INVALID_IDS_FILE = OUTPUT_DIR / "invalid_vk_ids.csv"
ERROR_IDS_FILE = OUTPUT_DIR / "error_vk_ids.csv"
FILTERED_TARGETS_FILE = OUTPUT_DIR / "targets_filtered.csv"


def read_existing_ids(path: Path, id_col: str = "vk_id") -> set[int]:
    if not path.exists():
        return set()

    try:
        df = pd.read_csv(path)
        if id_col not in df.columns:
            return set()

        ids = pd.to_numeric(df[id_col], errors="coerce").dropna().astype(int).tolist()
        return set(ids)
    except Exception:
        return set()


def append_row_to_csv(row: dict, path: Path):
    df_row = pd.DataFrame([row])
    write_header = not path.exists() or path.stat().st_size == 0
    df_row.to_csv(path, mode="a", header=write_header, index=False)


def profile_is_accessible(vk, user_id: int) -> tuple[bool, str]:
    """
    True  -> профиль доступен
    False -> закрыт / удалён / недоступен
    """
    try:
        response = vk.users.get(
            user_ids=str(user_id),
            fields="is_closed,can_access_closed"
        )

        if not response:
            return False, "empty_response"

        user = response[0]

        if user.get("deactivated"):
            return False, f"deactivated:{user.get('deactivated')}"

        if user.get("is_closed") and not user.get("can_access_closed", False):
            return False, "closed_profile"

        return True, "ok"

    except vk_api.exceptions.ApiError as e:
        # Частые коды недоступности профиля
        if getattr(e, "code", None) in (15, 18, 30):
            return False, f"api_error:{e.code}"
        raise


def collect_one_user_features(extractor: VKFeaturesExtractor, user_id: int) -> dict | None:
    features = extractor._extract_for_user(user_id)

    # если вернулся только user_id или вообще ничего — считаем, что сбор не удался
    if not features or len(features.keys()) <= 1:
        return None

    rename_map = {
        "user_id": "vk_id",
        "male_count": "male_friends",
        "female_count": "female_friends",
        "unknown_count": "unknown_friends",
    }

    row = {rename_map.get(k, k): v for k, v in features.items()}

    wanted_order = [
        "vk_id",
        "friends_count",
        "male_friends",
        "female_friends",
        "unknown_friends",
        "photo_count",
        "likes_total",
        "average_likes",
        "groups_count",
        "average_member",
    ]

    clean_row = {}
    for col in wanted_order:
        clean_row[col] = row.get(col, None)

    return clean_row


def save_filtered_targets(df_targets: pd.DataFrame):
    if not NODE_FEATURES_FILE.exists():
        return

    df_nodes = pd.read_csv(NODE_FEATURES_FILE)
    valid_ids = set(pd.to_numeric(df_nodes["vk_id"], errors="coerce").dropna().astype(int).tolist())

    df_targets_copy = df_targets.copy()
    df_targets_copy["vk_id"] = pd.to_numeric(df_targets_copy["vk_id"], errors="coerce")
    df_targets_copy = df_targets_copy.dropna(subset=["vk_id"]).copy()
    df_targets_copy["vk_id"] = df_targets_copy["vk_id"].astype(int)

    df_targets_copy = df_targets_copy[df_targets_copy["vk_id"].isin(valid_ids)].copy()
    df_targets_copy.to_csv(FILTERED_TARGETS_FILE, index=False)

In [8]:
def main():
    if not token:
        raise ValueError("ACCESS_TOKEN_VK не найден в token.env")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("1. Читаю JSON и извлекаю уникальных пользователей...")
    df_targets = load_targets_from_json(INPUT_JSON, TARGET_MAP)
    df_targets["vk_id"] = pd.to_numeric(df_targets["vk_id"], errors="coerce")
    df_targets = df_targets.dropna(subset=["vk_id"]).copy()
    df_targets["vk_id"] = df_targets["vk_id"].astype(int)

    # Сохраняем таргеты сразу
    targets_path = OUTPUT_DIR / "targets_ready.csv"
    df_targets.to_csv(targets_path, index=False)

    all_vk_ids = df_targets["vk_id"].drop_duplicates().tolist()

    success_ids = read_existing_ids(NODE_FEATURES_FILE, id_col="vk_id")
    invalid_ids = read_existing_ids(INVALID_IDS_FILE, id_col="vk_id")

    done_ids = success_ids | invalid_ids
    remaining_ids = [uid for uid in all_vk_ids if uid not in done_ids]

    print(f"Всего пользователей в таргетах: {len(all_vk_ids)}")
    print(f"Уже собрано успешно: {len(success_ids)}")
    print(f"Уже помечено как invalid: {len(invalid_ids)}")
    print(f"Осталось обработать: {len(remaining_ids)}")

    if not remaining_ids:
        print("Новых пользователей для обработки нет.")
        save_filtered_targets(df_targets)
        print(FILTERED_TARGETS_FILE)
        return

    extractor = VKFeaturesExtractor(users_id=remaining_ids)
    vk = extractor.vk

    processed_now = 0

    try:
        for idx, user_id in enumerate(remaining_ids, start=1):
            if idx % 50 == 0 or idx == 1:
                print(f"[{idx}/{len(remaining_ids)}] Обрабатываю vk_id={user_id}")

            try:
                accessible, reason = profile_is_accessible(vk, user_id)

                # закрытый / удалённый / недоступный профиль не добавляем в признаки
                if not accessible:
                    append_row_to_csv(
                        {
                            "vk_id": user_id,
                            "reason": reason,
                            "checked_at": datetime.utcnow().isoformat(),
                        },
                        INVALID_IDS_FILE,
                    )
                    continue

                row = collect_one_user_features(extractor, user_id)

                if row is None:
                    append_row_to_csv(
                        {
                            "vk_id": user_id,
                            "reason": "feature_extraction_failed",
                            "checked_at": datetime.utcnow().isoformat(),
                        },
                        ERROR_IDS_FILE,
                    )
                    continue

                # Сохраняем СРАЗУ после каждого пользователя
                append_row_to_csv(row, NODE_FEATURES_FILE)
                processed_now += 1

            except KeyboardInterrupt:
                print("Остановлено пользователем. Уже сохранённые данные не потеряются.")
                raise

            except Exception as e:
                logging.exception(f"Ошибка на user_id={user_id}: {e}")
                append_row_to_csv(
                    {
                        "vk_id": user_id,
                        "reason": f"exception:{type(e).__name__}",
                        "checked_at": datetime.utcnow().isoformat(),
                    },
                    ERROR_IDS_FILE,
                )
                continue

    finally:
        # На каждом завершении обновляем таргеты только для тех, по кому есть признаки
        save_filtered_targets(df_targets)

    print("Готово.")
    print(f"Новых успешно собранных пользователей в этом запуске: {processed_now}")
    print(targets_path)
    print(NODE_FEATURES_FILE)
    print(INVALID_IDS_FILE)
    print(ERROR_IDS_FILE)
    print(FILTERED_TARGETS_FILE)


if __name__ == "__main__":
    main()

1. Читаю JSON и извлекаю уникальных пользователей...
Всего пользователей в таргетах: 6987
Уже собрано успешно: 76
Уже помечено как invalid: 32
Осталось обработать: 6879
[1/6879] Обрабатываю vk_id=476029973


/var/folders/jr/6ytflvns1_n0vd_mpbq_x8mm0000gn/T/ipykernel_42011/2802284879.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "checked_at": datetime.utcnow().isoformat(),
/var/folders/jr/6ytflvns1_n0vd_mpbq_x8mm0000gn/T/ipykernel_42011/2802284879.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "checked_at": datetime.utcnow().isoformat(),
/var/folders/jr/6ytflvns1_n0vd_mpbq_x8mm0000gn/T/ipykernel_42011/2802284879.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "checked_at": datetime.utcnow().isoformat(),
/

Остановлено пользователем. Уже сохранённые данные не потеряются.


KeyboardInterrupt: 